In [8]:
# =========================================================
# 0. IMPORT & SETUP
# =========================================================
import moxing as mox
import pandas as pd
import numpy as np
import mindspore as ms
from mindspore import Tensor, load_checkpoint, load_param_into_net
from sklearn.preprocessing import MinMaxScaler
import time

from transformers import LSTMTransformer

ms.set_context(mode=ms.PYNATIVE_MODE)

# =========================================================
# 1. PATH CONFIGURATION
# =========================================================
obs_model_path = 'obs://mindspore-dataset-1/output_model2/Transformer/lstm_transformer.ckpt'
obs_sensor_path = 'obs://mindspore-dataset-1/Data/sensor_terkini.csv'
obs_prediction_path = 'obs://mindspore-dataset-1/Prediction/transformer.csv'
obs_history_path = 'obs://mindspore-dataset-1/Data/data_bersih.csv' # Basis data training

local_model_path = './model.ckpt'
local_sensor_path = './sensor_terkini.csv'
local_output_path = './transformer_results.csv'
local_history_path = './data_bersih.csv'

print("Download model dari OBS...")
mox.file.copy(obs_model_path, local_model_path)

# =========================================================
# 2. INISIALISASI SCALER (MENGGUNAKAN BASIS DATA TRAINING)
# =========================================================
# Membaca data historis agar range Min-Max Scaler SAMA PERSIS dengan saat training
print("Mengunduh data riwayat untuk inisialisasi basis objek scaler...")
mox.file.copy(obs_history_path, local_history_path)
df_hist = pd.read_csv(local_history_path)

# Pembersihan data historis dari NaN (Sesuai prosedur training)
df_hist[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']] = df_hist[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']].interpolate(method='linear', limit_direction='both')
df_hist[['Inline', 'Exhaust', 'Heater']] = df_hist[['Inline', 'Exhaust', 'Heater']].interpolate(method='linear', limit_direction='both')
df_hist = df_hist.ffill().bfill()

# Scaler dicari polanya (.fit) dari data seluruhnya, BUKAN dari data realtime yang sepotong-sepotong
scaler_features = MinMaxScaler()
scaler_features.fit(df_hist[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']].values)

scaler_targets = MinMaxScaler()
scaler_targets.fit(df_hist[['Inline', 'Exhaust', 'Heater']].values)

# =========================================================
# 3. LOAD MODEL DENGAN PARAMETER TRAINING
# =========================================================
# Parameter wajib kembar dengan konfigurasi PARAMS saat training
PARAMS = {
    'input_dim': 5,         # 5 Fitur Sensor + 3 Fitur Status Aktuator Sebelumnya
    'lstm_units': 128,       
    'num_heads': 4,         
    'ff_dim': 256,          
    'dropout_rate': 0.15,   # Nilai dropout tidak mempengaruhi inferensi (set_train=False)
    'seq_length': 18        # Harus 24 sesuai setelan sekuens training terakhir
}

model = LSTMTransformer(
    input_dim=PARAMS['input_dim'],
    lstm_units=PARAMS['lstm_units'],
    num_heads=PARAMS['num_heads'],
    ff_dim=PARAMS['ff_dim'],
    dropout_rate=PARAMS['dropout_rate']
)

# Memuat bobot network
param_dict = load_checkpoint(local_model_path)
load_param_into_net(model, param_dict)
model.set_train(False)

print("✅ Model dengan 5 input sensor berhasil di-load!\n")

# =========================================================
# 4. REAL-TIME PREDICTION LOOP (5 INPUT SENSOR)
# =========================================================
print("Memulai loop inferensi otomatis...")
try:
    while True:
        # Ambil berkas sensor dari OBS Huawei Cloud
        mox.file.copy(obs_sensor_path, local_sensor_path)
        df_now = pd.read_csv(local_sensor_path)

        # Antisipasi jika ada data kosong pada stream IoTDA
        df_now = df_now.interpolate(method='linear', limit_direction='both').ffill().bfill()

        # Ambil murni 5 kolom sensor lingkungan
        fitur_cols = ['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']
        features_raw = df_now[fitur_cols].values

        # PERBAIKAN UTAMA: Gunakan .transform(), JANGAN .fit_transform() di dalam loop
        features_scaled = scaler_features.transform(features_raw)

        X_input = []
        # Cek ketersediaan panjang baris data streaming
        if len(features_scaled) >= PARAMS['seq_length']:
            # Ambil sekuens jendela waktu terakhir sebanyak SEQ_LENGTH
            X_input.append(features_scaled[-PARAMS['seq_length']:])
            X_input = np.array(X_input, dtype=np.float32)

            # Prediksi dengan MindSpore
            pred_scaled = model(Tensor(X_input, ms.float32)).asnumpy()

            # Denormalisasi output ke nilai satuan asli (Persen / Status Saklar)
            pred_original = scaler_targets.inverse_transform(pred_scaled)

            # Pembatasan logika kontrol fisik aktuator
            inline_val = int(np.clip(np.round(pred_original[0, 0] / 10) * 10, 0, 100))
            exhaust_val = int(np.clip(np.round(pred_original[0, 1] / 10) * 10, 0, 100))
            heater_val = int(np.clip(np.round(pred_original[0, 2]), 0, 1))

            # Bungkus ke dataframe output
            result_df = pd.DataFrame({
                'inline_pred': [inline_val],
                'exhaust_pred': [exhaust_val],
                'heater_pred': [heater_val]
            })

            # Tulis lokal dan kirim balik ke OBS untuk dikonsumsi dashboard Flask/Web
            result_df.to_csv(local_output_path, index=False)
            mox.file.copy(local_output_path, obs_prediction_path)

            print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Prediksi Diperbarui:")
            print(f" -> Rekomendasi Inline  : {inline_val} %")
            print(f" -> Rekomendasi Exhaust : {exhaust_val} %")
            print(f" -> Rekomendasi Heater  : {'ON' if heater_val == 1 else 'OFF'}")
            print("-" * 45)

        else:
            print(f"⚠️ Jumlah baris data di OBS ({len(features_scaled)} baris) belum memenuhi syarat SEQ_LENGTH ({PARAMS['seq_length']}).")
        
        time.sleep(2)

except KeyboardInterrupt:
    print("\n🛑 Loop prediksi dihentikan secara aman oleh user.")

Download model dari OBS...
Mengunduh data riwayat untuk inisialisasi basis objek scaler...
✅ Model dengan 5 input sensor berhasil di-load!

Memulai loop inferensi otomatis...
[2026-05-27 00:00:35] Prediksi Diperbarui:
 -> Rekomendasi Inline  : 40 %
 -> Rekomendasi Exhaust : 60 %
 -> Rekomendasi Heater  : OFF
---------------------------------------------
[2026-05-27 00:00:37] Prediksi Diperbarui:
 -> Rekomendasi Inline  : 40 %
 -> Rekomendasi Exhaust : 60 %
 -> Rekomendasi Heater  : OFF
---------------------------------------------
[2026-05-27 00:00:39] Prediksi Diperbarui:
 -> Rekomendasi Inline  : 40 %
 -> Rekomendasi Exhaust : 60 %
 -> Rekomendasi Heater  : OFF
---------------------------------------------
[2026-05-27 00:00:41] Prediksi Diperbarui:
 -> Rekomendasi Inline  : 40 %
 -> Rekomendasi Exhaust : 60 %
 -> Rekomendasi Heater  : OFF
---------------------------------------------
[2026-05-27 00:00:44] Prediksi Diperbarui:
 -> Rekomendasi Inline  : 40 %
 -> Rekomendasi Exhaust : 6